#“Reload model + rerun Step 7”

In [ ]:
import torch
import torch.nn as nn

class MaskedTimeTransformer(nn.Module):
    def __init__(self, K, window=128, d_model=128, nhead=4, num_layers=3, dim_ff=256, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(K, d_model)
        self.pos_emb = nn.Parameter(torch.randn(window, d_model) * 0.02)

        layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, K)

    def forward(self, x):
        B, L, _ = x.shape
        h = self.in_proj(x) + self.pos_emb[:L].unsqueeze(0)
        h = self.encoder(h)
        return self.out_proj(h)


In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

K = 40          # set this to your node count used in training
window = 128    # must match the model positional embedding length

model = MaskedTimeTransformer(K=K, window=window).to(device)
model.load_state_dict(torch.load("run1_clean_model.pt", map_location=device))
model.eval()

print("Loaded model on:", device)


In [ ]:
# ---- Missing definitions required for Step 7 reruns (add once) ----
import numpy as np
import torch
from torch.utils.data import Dataset

def destroy_bitflip_X(X, delta, seed=0):
    """Flip each bit independently with probability delta."""
    rng = np.random.default_rng(seed)
    flips = rng.random(X.shape) < delta
    X_tilde = X.copy()
    X_tilde[flips] = 1 - X_tilde[flips]
    return X_tilde

class MaskedWindowDataset(Dataset):
    """Sliding windows over Z with per-entry random masks."""
    def __init__(self, Z, window=128, mask_ratio=0.15, seed=123):
        rng = np.random.default_rng(seed)
        T, K = Z.shape
        self.Z = Z.astype(np.float32)
        self.window = window
        self.N = T - window
        self.masks = rng.random((self.N, window, K)) < mask_ratio

    def __len__(self):
        return self.N

    def __getitem__(self, idx):
        x = self.Z[idx:idx+self.window].copy()   # (L,K)
        m = self.masks[idx]                      # (L,K) bool
        y = x.copy()
        x[m] = 0.0
        return torch.from_numpy(x), torch.from_numpy(m), torch.from_numpy(y)

def baseline_predict_window_node_mean(x_masked_np):
    """Predict each node by its mean over time within the (masked) window."""
    node_means = x_masked_np.mean(axis=1, keepdims=True)  # (B,1,K)
    return np.repeat(node_means, x_masked_np.shape[1], axis=1)


#Step 1:

In [ ]:
import numpy as np

def simulate_global_state(
        T=100_000,
        D=256,
        mu=-1.5,
        sigma=0.3,
        sigma_drift=0.001,
        seed=42
):
    rng = np.random.default_rng(seed)

    # base logits for each bit
    alpha = rng.normal(mu, sigma, size=D)
    beta = rng.normal(mu, sigma, size=D)

    # drift term
    eps = np.zeros(D)

    # state matrix
    X = np.zeros((T, D), dtype=np.int8)

    #initial state
    X[0] = rng.binomial(1, 0.5, size=D)

    for t in range(1, T):
        # update drift
        eps += rng.normal(0, sigma_drift, size=D)

        # Transition probabilities
        p = 1 / (1 + np.exp(-(alpha + eps)))
        q = 1 / (1 + np.exp(-(beta+eps)))

        prev = X[t-1]

        # transition
        prob_1 = np.where(prev == 0, p, 1 - q)
        X[t] = rng.binomial(1, prob_1)
        
    return X

#step 2:

In [ ]:
import numpy as np

def construct_nodes(
        D=256,
        K=40,
        subset_size=25,
        overlap_target=5,
        seed=123
):
    """
    Construct K subsets of size 25 from {1,...,D} with controlled overlap.
    """
    rng = np.random.default_rng(seed)
    subsets = []

    # first subset purely random
    first = rng.choice(D, subset_size, replace=False)
    subsets.append(set(first))

    for _ in range(1, K):
        # pick overlap_target indices from an existing subset
        base = rng.choice(list(subsets[rng.integers(len(subsets))]),
                          overlap_target,
                          replace=False)
        remaining = subset_size - overlap_target
        rest = rng.choice(
            list(set(range(D)) - set(base)),
            remaining,
            replace=False
        )
        subsets.append(set(base) | set(rest))

    return subsets

def compute_node_signals(X, subsets):
    """
    X: (T, D)
    subsets: list of index sets
    returns Z: (T, K)
    """
    T = X.shape[0]
    K = len(subsets)
    Z = np.zeros((T, K))
    for k, S in enumerate(subsets):
        Z[:, k] = X[:, list(S)].mean(axis=1)
    return Z

# ----- apply to previously simulated X for sanity check -----
X = simulate_global_state(T=50_000, seed=123)

subsets = construct_nodes()
Z = compute_node_signals(X, subsets)

# sanity checks
Z.shape, np.mean(np.std(Z, axis=0)), np.corrcoef(Z.T)[:5, :5]

#step 4:

In [ ]:
import numpy as np

# ----- minimal regeneration -----
def simulate_global_state(
    T=10_000,
    D=356,
    seed=123
):
    rng = np.random.default_rng(seed)
    alpha = rng.normal(-1.5, 0.3, size=D)
    beta = rng.normal(-1.5, 0.3, size=D)
    eps = np.zeros(D)
    X = np.zeros((T, D), dtype=np.int8)
    X[0] = rng.binomial(1, 0.5, size=D)
    for t in range(1, T):
        eps += rng.normal(0, 0.001, size=D)
        p = 1 / (1 + np.exp(-(alpha + eps)))
        q = 1 / (1 + np.exp(-(beta + eps)))
        X[t] = rng.binomial(1, np.where(X[t-1] == 0, p, 1-q))
    return X

def construct_nodes(
    D=256,
    K=40,
    subset_size=25,
    overlap_target=5,
    seed=123
):
    rng = np.random.default_rng(seed)
    subsets = [set(rng.choice(D, subset_size, replace=False))]
    for _ in range(1, K):
        base = rng.choice(list(subsets[rng.integers(len(subsets))]),
        overlap_target, replace=False)
        rest = rng.choice(list(set(range(D)) - set(base)),
        subset_size - overlap_target, replace=False)
        subsets.append(set(base) | set(rest))
    return subsets

def compute_node_signals(X, subsets):
    Z = np.zeros((X.shape[0], len(subsets)), dtype=np.float32)
    for k, S in enumerate(subsets):
        Z[:, k] = X[:, list(S)].mean(axis=1)
    return Z

def prepare_masked_sequences(Z, window=128, mask_ratio=0.15, seed=123):
    rng = np.random.default_rng(seed)
    T, K = Z.shape
    N = T - window
    X_masked = np.zeros((N, window, K), dtype=np.float32)
    mask = np.zeros((N, window, K), dtype=bool)

    for i in range(N):
        X_masked[i] = Z[i:i+window]
        m = rng.random((window, K)) < mask_ratio
        mask[i] = m
        X_masked[i][m] = 0.0

    return X_masked, mask

# --- run ---
X = simulate_global_state()
subsets = construct_nodes()
Z = compute_node_signals(X, subsets)

X_masked, mask = prepare_masked_sequences(Z)
X_masked.shape, mask.mean()

#step 5:


import numpy as np

# ---------- step 1-2 helper code ----------
def simulate_global_state(
    T=20_000,
    D=256,
    seed=123,
    mu=-1.5,
    sigma=0.3,
    sigma_drift=0.001
):
    rng = np.random.default_rng(seed)
    alpha = rng.normal(mu, sigma, size=D)
    beta = rng.normal(mu, sigma, size=D)
    eps = np.zeros(D)
    X = np.zeros((T, D), dtype=np.int8)
    X[0] = rng.binomial(1, 0.5, size=D)
    for t in range(1, T):
        eps += rng.normal(0, sigma_drift, size=D)
        p = 1 / (1 + np.exp(-(alpha + eps)))
        q = 1 / (1 + np.exp(-(beta + eps)))
        X[t] = rng.binomial(1, np.where(X[t-1]==0, p, 1-q))
    return X

def construct_nodes(
    D=256,
    K=40,
    subset_size=25,
    overlap_target=5,
    seed=123
):
    rng = np.random.default_rng(seed)
    subsets = [set(rng.choice(D, subset_size, replace=False))]
    for _ in range(1, K):
        base_subset = subsets[rng.integers(len(subsets))]
        base = rng.choice(list(base_subset), overlap_target, replace=False)
        rest = rng.choice(list(set(range(D)) - set(base)), subset_size - overlap_target, replace=False)
        subsets.append(set(base) | set(rest))
    return subsets

def compute_node_signals(X, subsets):
    Z = np.zeros((X.shape[0], len(subsets)), dtype=np.float32)
    for k, S in enumerate(subsets):
        Z[:, k] = X[:, list(S)].mean(axis=1)
    return Z

# ---------- step 5: Destruction operators ----------
def destroy_bitflip_X(X, delta, seed=0):
    rng = np.random.default_rng(seed)
    flips = rng.random(X.shape) < delta
    X_tilde = X.copy()
    X_tilde[flips] = 1 - X_tilde[flips]
    return X_tilde

def destroy_dropout_Z(Z, delta, seed=0):
    rng = np.random.default_rng(seed)
    drop = rng.random(Z.shape) < delta
    Z_tilde = Z.copy()
    Z_tilde[drop] = 0.0
    return Z_tilde

def destroy_shuffle_time(Z, delta, seed=0):
    rng = np.random.default_rng(seed)
    T = Z.shape[0]
    m = int(delta * T)
    if m<= 1:
        return Z.copy()
    idx = np.arange(T)
    pick = rng.choice(T, size=m, replace=False)
    perm = pick.copy()
    rng.shuffle(perm)
    idx[pick] = idx[perm]
    return Z[idx].copy()

# ----- diagnostics -----
def corr_summary(Z):
    W = np.corrcoef(Z.T)
    W = np.nan_to_num(W)
    K = W.shape[0]
    offdiag = W[~np.eye(K, dtype=bool)]
    return float(offdiag.mean()), float(np.median(np.abs(offdiag))), float(np.max(np.abs(offdiag)))

def lag1_corr_mean(Z):
    Z0, Z1 = Z[:-1], Z[1:]
    c = []
    for k in range(Z.shape[1]):
        a, b = Z0[:, k], Z1[:, k]
        sa, sb = np.std(a), np.std(b)
        if sa < 1e-8 or sb < 1e-8:
            c.append(0.0)
        else:
            c.append(np.corrcoef(a, b)[0, 1])
    return float(np.mean(c))

# ----- run -----
X = simulate_global_state(T=20_000, seed=123)
subsets = construct_nodes(seed=123)
Z = compute_node_signals(X, subsets)

# ------ apply bit-flip destruction and observe effects on Z ----
deltas = [0.0, 0.05, 0.10, 0.20, 0.30]
report_bitflip = []
for d in deltas:
    Xd = destroy_bitflip_X(X, d, seed=999)
    Zd = compute_node_signals(Xd, subsets)
    cmean, absmed, absmax, = corr_summary(Zd)
    report_bitflip.append({
        "delta": d,
        "lag1_corr_mean": lag1_corr_mean(Zd),
        "crosscorr_abs_median": absmed,
        "crosscorr_abs_max": absmax,
        "Z_std_mean": float(np.mean(np.std(Zd, axis=0))),
    })

# ------ compare alternative destruction modes on Z directly ----
d = 0.20
Z_drop = destroy_dropout_Z(Z, d, seed=999)
Z_shuf = destroy_shuffle_time(Z, d, seed=999)

compare_modes = {
    "base": {
        "lag1_corr_mean": lag1_corr_mean(Z),
        "crosscorr_abs_median": corr_summary(Z)[1],
        "crosscorr_abs_max": corr_summary(Z)[2],
    },
    "dropout_Z (delta=0.20)": {
        "lag1_corr_mean": lag1_corr_mean(Z_drop),
        "crosscorr_abs_median": corr_summary(Z_drop)[1],
        "crosscorr_abs_max": corr_summary(Z_drop)[2],
    },
    "shuffle_time (delta=0.20)": {
        "lag1_corr_mean": lag1_corr_mean(Z_shuf),
        "crosscorr_abs_median": corr_summary(Z_shuf)[1],
        "crosscorr_abs_max": corr_summary(Z_shuf)[2],
    }
}

report_bitflip, compare_modes

#Step 7 utilities:

In [ ]:
# ---- 7.3 discretization ----
def discretize(x, bins):
    return np.digitize(x, bins) -1

In [ ]:
# ---- 7.4 Entropy utilities -----
import numpy as np

def entropy_from_counts(counts):
    p = counts / counts.sum()
    p = p[p > 0]
    return -np.sum(p * np.log2(p))

def joint_entropy(x, y):
    xy = np.stack([x, y], axis=1)
    _, counts = np.unique(xy, axis=0, return_counts=True)
    return entropy_from_counts(counts)

def conditional_entropy(x, y): 
    # H(x|y) = H(x,y) - H(y)
    _, cy = np.unique(y, return_counts=True)
    return joint_entropy(x, y) - entropy_from_counts(cy)

def mutual_information(x, y):
    _, cx = np.unique(x, return_counts=True)
    return entropy_from_counts(cx) - conditional_entropy(x, y)

In [ ]:
# ----- 7.5: collect masked predictions -----
@torch.no_grad()
def collect_masked_predictions(model, loader, device):
    model.eval()
    z_true, z_pred = [], []

    for xb, mb, yb in loader:
        xb = xb.to(device).float()
        mb = mb.numpy().astype(bool)
        yb = yb.numpy()

        yhat = model(xb).cpu().numpy()

        z_true.append(yb[mb])
        z_pred.append(yhat[mb])

    return np.concatenate(z_true), np.concatenate(z_pred)

# and for baseline (window-node-mean) we have:
def collect_baseline_predictions(loader):
    z_true, z_pred = [], []

    for xb, mb, yb in loader:
        x = xb.numpy()
        m = mb.numpy().astype(bool)
        y = yb.numpy()

        yhat = baseline_predict_window_node_mean(x)

        z_true.append(y[m])
        z_pred.append(yhat[m])
    
    return np.concatenate(z_true), np.concatenate(z_pred)

In [ ]:
# ----- 7.6: compute information metrics -------
bins = np.linspace(0, 1, 21)

z_t, z_p = collect_masked_predictions(model, val_loader, device)

zt_d = discretize(z_t, bins)
zp_d = discretize(z_p, bins)

H_cond = conditional_entropy(zt_d, zp_d)
MI = mutual_information(zt_d, zp_d)

print("H(Z | Z_hat) =", H_cond)
print("I(Z ; Z_hat) =", MI)
print("Normalized retention =", MI / entropy_from_counts(np.unique(zt_d, return_counts=True)[1]))

In [ ]:
# ------- 7.7: delta sweep ------- 
## ------ phase transitions part -----
# in this stage, we repeat steps 5 to 7 for increasing delta


# constants reused from Step 6
window = 128
mask_ratio = 0.15
split = int(0.8 * Z.shape[0])

results = []

for delta in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50]:
    Xd = destroy_bitflip_X(X, delta, seed=999)
    Zd = compute_node_signals(Xd, subsets)

# rebuild validation loader with same window and mask
Z_val_d = Zd[split - window:]
val_ds_d = MaskedWindowDataset(Z_val_d, window=window, mask_ratio=mask_ratio, seed=2)
val_loader_d = DataLoader(val_ds_d, batch_size=64, shuffle=False)

zt, zp = collect_masked_predictions(model, val_loader_d, device)
zt_d = discretize(zt, bins)
zp_d = discretize(zp, bins)

MI = mutual_information(zt_d, zp_d)

results.append({
    "delta": delta,
    "mutual_information": MI
})

In [ ]:
# --- helper functions to collect baseline masked predictions ----
import numpy as np
import torch
from torch.utils.data import DataLoader
import json
import matplotlib.pyplot as plt

def collect_baseline_window_predictions(val_loader):
    z_true, z_pred = [], []
    for xb, mb, yb in val_loader:
        x = xb.numpy().astype(np.float32)
        m = mb.numpy().astype(bool)
        y = yb.numpy().astype(np.float32)

        yhat = baseline_predict_window_node_mean(x) # (B, L, K)
        z_true.append(y[m])
        z_pred.append(yhat[m])

    return np.concatenate(z_true), np.concatenate(z_pred)

In [ ]:
# ----- sweep: compute ml + retention for trasformer and baseline ----
def H_from_samples(discrete_samples):
    _, counts = np.unique(discrete_samples, return_counts=True)
    return entropy_from_counts(counts)

results = []

for delta in [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    # 1) destroy world -> recompute node series
    Xd = destroy_bitflip_X(X, delta, seed=999)
    Zd = compute_node_signals(Xd, subsets)

    # 2) build val loader on destroyed data (same protocol as step 6/7)
    Z_val_d = Zd[split-window:] # keep overlap so first window is valid
    val_ds_d = MaskedWindowDataset(Z_val_d, window=window, mask_ratio=mask_ratio, seed=2)
    val_loader_d = DataLoader(val_ds_d, batch_size=64, shuffle=False, num_workers=0)

    # 3) Transformer masked samples
    zt_tf, zp_tf = collect_masked_predictions(model, val_loader_d, device)
    
    zt_tf_d = discretize(zt_tf, bins)
    zp_tf_d = discretize(zp_tf, bins)

    H_true = H_from_samples(zt_tf_d)
    MI_tf = mutual_information(zt_tf_d, zp_tf_d)
    R_tf = (MI_tf / H_true) if H_true > 1e-12 else 0

    # 4) baseline masked samples (window-node-mean)
    zt_b, zp_b = collect_baseline_window_predictions(val_loader_d)
    zt_b_d = discretize(zt_b, bins)
    zp_b_d = discretize(zp_b, bins)

    # H_true should match (up to sampling noise); reuse H_true from transformer samples
    MI_b = mutual_information(zt_b_d, zp_b_d)
    R_b = (MI_b / H_true) if H_true > 1e-12 else 0.0

    results.append({
        "delta": float(delta),
        "H_true": float(H_true),
        "MI_transformer": float(MI_tf),
        "R_transformer": float(R_tf),
        "MI_baseline_window": float(MI_b),
        "R_baseline_window": float(R_b),
    })

    results[:2], results[-1]

In [ ]:
# --- save results in json file -----
with open("step7_delta_sweep_info.json", "w") as f:
    json.dump(results, f, indent=2)
print("Saved: step7_delta_sweep_info_metrics.json")

In [ ]:
# ------ plot MI vs Delta & Retention vs Delta
# MI plot
deltas = [r["delta"] for r in results]
mi_tf = [r["MI_transformer"] for r in results]
mi_b = [r["MI_baseline_window"] for r in results]

plt.figure()
plt.plot(deltas, mi_tf, marker="o")
plt.plot(deltas, mi_b, marker="o")
plt.xlabel("Destruction rate δ")
plt.ylabel("mutual information I(Z ; Y) [bits]")
plt.title("Mutual Information vs Destruction")
plt.legend(["transformer", "baseline (window mean)"])
plt.show()

In [ ]:
# Retention plot R(delta) = I / H
r_tf = [r["R_transformer"] for r in results]
r_b = [r["R_baseline_window"] for r in results]

plt.figure()
plt.plot(deltas, r_tf, marker="o")
plt.plot(deltas, r_b, marker="o")
plt.xlabel("destruction rate δ")
plt.ylabel("retention R(δ) = I/H")
plt.title("Information retention vs destruction")
plt.legend(["transformer", "baseline (window mean)"])
#plt.ylim(-0.05, 1.05)
plt.ylim(0, 0.03)
plt.show()

In [ ]:
# ----- 1) failure when transformer advantage disappears ----
def failure_threshold_gap(results, alpha=0.2, use_key="R"):
    # use_key can be "R" or "MI"
    tf_key = f"{use_key}_transformer"
    b_key = f"{use_key}_baseline_window"

    deltas = np.array([r["delta"] for r in results], dtype=float)
    gap = np.array([r[tf_key] - r[b_key] for r in results], dtype=float)

    gap0 = gap[0]
    target = alpha * gap0

    # find first crossing 
    for i in range(1, len(deltas)):
        if gap[i] <= target:
            # interpolate between i-1 and i
            d0, d1 = deltas[i-1], deltas[i]
            g0, g1 = gap[i-1], gap[i]
            if g1 == g0:
                return float(d1)
            return float(d0 + (target - g0) * (d1 - d0) / (g1 - g0))
        
    return None # never crossed in your grid

In [ ]:
delta_star = failure_threshold_gap(results, alpha=0.2, use_key="R")
print("delta* (gap-retention, 20% rule):", delta_star)

In [ ]:
# ------ 2) Failure when retention falls below an absolute floor ----
def failure_threshold_absolute(results, eps=0.01):
    deltas = np.array([r["delta"] for r in results], dtype=float)
    Rtf = np.array([r["R_transformer"] for r in results], dtype=float)

    for i in range(1, len(deltas)):
        if Rtf[i] <= eps:
            d0, d1 = deltas[i-1], deltas[i]
            r0, r1 = Rtf[i-1], Rtf[i]
            if r1 == r0:
                return float(d1)
            return float(d0 + (eps - r0) * (d1- d0) / (r1 - r0))
    return None

In [ ]:
delta_star_abs = failure_threshold_absolute(results, eps=0.01)
print("delta* (R <= 1%):", delta_star_abs)

In [ ]:
# ------ 3) failure at the "knee point" (phase transition) ----
def failure_threshold_knee(results, key="R_transformer"):
    deltas = np.array([r["delta"] for r in results], dtype=float)
    y = np.array([r[key] for r in results], dtype=float)

    # need at least 3 points
    if len(y) < 3:
        return None
    
    # second diffrence (assumes roughly uniform delta spacing)
    second = y[:-2] - 2*y[1:-1] + y[2:]
    idx = np.argmin(second) + 1 # most negative curvature (fastest drop)
    return float(deltas[idx])

In [ ]:
delta_star_knee = failure_threshold_knee(results, key="R_transformer")
print("delta* (knee):", delta_star_knee)

In [ ]:
import json 
from datetime import datetime

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
fname = f"step7_results_delta_0p00_to_0p30_{stamp}.json"

with open(fname, "w") as f:
    json.dump(results, f, indent=2)

print("Saved:", fname)

In [ ]:
import csv

csv_name = fname.replace(".json", ".csv")
keys = results[0].keys()

with open(csv_name, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=keys)
    w.writeheader()
    w.writerows(results)

print("saved:", csv_name)

In [ ]:
run = {
    "config": {
        # model
        "K": K,
        "window": window,
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_ff": 256,
        "dropout": 0.1,

        # training
        "mask_ratio": mask_ratio,
        "train_val_split": 0.8,

        # evaluation
        "bins": bins.tolist(),
        "deltas": deltas,

        # destruction
        "destruction": "bitflip_X",
        "seed_destroy": 999
    },
    "results": results
}


In [ ]:
import json
import torch

with open("step7_run_YYYYMMDD_HHMMSS.json", "r") as f:
    run = json.load(f)

cfg = run["config"]

model = MaskedTimeTransformer(
    K=cfg["K"],
    window=cfg["window"],
    d_model=cfg["d_model"],
    nhead=cfg["nhead"],
    num_layers=cfg["num_layers"],
    dim_ff=cfg["dim_ff"],
    dropout=cfg["dropout"]
)

model.load_state_dict(torch.load("run1_clean_model.pt"))
model.eval()


In [ ]:
import numpy as np
from torch.utils.data import DataLoader

# ---- (A) Re-generate base world + nodes exactly as before ----
# IMPORTANT: use the SAME seeds/hyperparameters as your original run
T = 20000                 # use the same T as training if you did
seed_world = 123
seed_nodes = 123

X = simulate_global_state(T=T, seed=seed_world)
subsets = construct_nodes(K=K, seed=seed_nodes)
Z = compute_node_signals(X, subsets)

# split must match Step 6 convention
split = int(0.8 * Z.shape[0])
mask_ratio = 0.15

# discretization bins (fixed)
bins = np.linspace(0, 1, 21)

# choose a NEW delta range here
deltas = np.arange(0.0, 0.55, 0.05)  # 0.00 ... 0.50

results = []

for delta in deltas:
    # destroy X -> compute Zd
    Xd = destroy_bitflip_X(X, delta, seed=999)
    Zd = compute_node_signals(Xd, subsets)

    # build val loader on destroyed data
    Z_val_d = Zd[split - window:]
    val_ds_d = MaskedWindowDataset(Z_val_d, window=window, mask_ratio=mask_ratio, seed=2)
    val_loader_d = DataLoader(val_ds_d, batch_size=64, shuffle=False, num_workers=0)

    # transformer masked samples
    zt_tf, zp_tf = collect_masked_predictions(model, val_loader_d, device)
    zt_tf_d = discretize(zt_tf, bins)
    zp_tf_d = discretize(zp_tf, bins)

    H_true = H_from_samples(zt_tf_d)
    MI_tf = mutual_information(zt_tf_d, zp_tf_d)
    R_tf = (MI_tf / H_true) if H_true > 1e-12 else 0.0

    # baseline masked samples
    zt_b, zp_b = collect_baseline_window_predictions(val_loader_d)
    zt_b_d = discretize(zt_b, bins)
    zp_b_d = discretize(zp_b, bins)

    MI_b = mutual_information(zt_b_d, zp_b_d)
    R_b = (MI_b / H_true) if H_true > 1e-12 else 0.0

    results.append({
        "delta": float(delta),
        "H_true": float(H_true),
        "MI_transformer": float(MI_tf),
        "R_transformer": float(R_tf),
        "MI_baseline_window": float(MI_b),
        "R_baseline_window": float(R_b),
    })

results[:3], results[-1]
